In [1]:
import pandas as pd
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("mehmetisik/amazon-review")
print("Path to dataset files:", path)

# Load data
data = pd.read_csv(path + '/amazon_review.csv')

100%|███████████████████████████████████████████████████████████████████████████████| 705k/705k [00:00<00:00, 14.0MB/s]

Extracting files...


Path to dataset files: C:\Users\Nizam\.cache\kagglehub\datasets\mehmetisik\amazon-review\versions\1


In [2]:
#Ensure text column is string
data['reviewText'] = data['reviewText'].astype(str)
data.head(3)

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote
0,A3SBTW3WS4IQSN,B007WTAJTO,NaN,"[0, 0]",No issues.,4.0,Four Stars,1406073600,2014-07-23,138,0,0
1,A18K1ODH1I2MVB,B007WTAJTO,0mie,"[0, 0]","Purchased this for my device, it worked as adv...",5.0,MOAR SPACE!!!,1382659200,2013-10-25,409,0,0
2,A2FII3I2MBMUIA,B007WTAJTO,1K3,"[0, 0]",it works as expected. I should have sprung for...,4.0,nothing to really say....,1356220800,2012-12-23,715,0,0


# Preprocessing of data

In [3]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords if you haven't already
nltk.download('stopwords')

def preprocess_sentence(sentence):
    "Cleans text by removing numbers, punctuation, converting to lowercase, and removing stop words."

    # Remove numbers
    sentence = re.sub(r'\d+', '', sentence)

    # Remove punctuation
    sentence = re.sub(r'[^\w\s]', '', sentence)

    # Convert to lowercase
    sentence = sentence.lower()

    # Remove stop words
    stop_words = set(stopwords.words('english'))
    words = sentence.split()
    sentence = ' '.join([word for word in words if word not in stop_words])

    return sentence

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Nizam\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


# TextBlob for sentiment extraction

In [5]:
!pip install textblob

   ---------------------------------------- 0.0/624.3 kB ? eta -:--:--
   --------------------------------------- 624.3/624.3 kB 10.1 MB/s eta 0:00:00


In [6]:
from textblob import TextBlob
# Apply preprocessing
data['processed_sentence'] = data['reviewText'].apply(preprocess_sentence)

# Create Sentiment Labels using TextBlob
def classify(polarity):
    if polarity > 0:
        return "positive"
    elif polarity < 0:
        return "negative"
    else:
        return "neutral"

data["sentiment_score"] = data["processed_sentence"].apply(lambda x: TextBlob(x).sentiment.polarity)
data["sentiment_label"] = data["sentiment_score"].apply(classify)

In [7]:
data[['reviewText','processed_sentence','sentiment_score','sentiment_label']]

,reviewText,processed_sentence,sentiment_score,sentiment_label
0,No issues.,issues,0.000000,neutral
1,"Purchased this for my device, it worked as adv...",purchased device worked advertised never much ...,-0.100000,negative
2,it works as expected. I should have sprung for...,works expected sprung higher capacity think ma...,0.129167,positive
3,This think has worked out great.Had a diff. br...,think worked greathad diff bran gb card went s...,0.250000,positive
4,"Bought it with Retail Packaging, arrived legit...",bought retail packaging arrived legit orange e...,0.386667,positive
...,...,...,...,...
4910,I bought this Sandisk 16GB Class 10 to use wit...,bought sandisk gb class use htc inspire months...,0.012500,positive
4911,Used this for extending the capabilities of my...,used extending capabilities samsung galaxy not...,0.800000,positive
4912,Great card that is very fast and reliable. It ...,great card fast reliable comes optional adapte...,0.350000,positive
4913,Good amount of space for the stuff I want to d...,good amount space stuff want fits gopro say,0.700000,positive


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

feature = data['processed_sentence']
target = data['sentiment_label']

# Split into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(feature, target, test_size=0.2, random_state=42)

# Extract features using TF-IDF
vectorizer = TfidfVectorizer()
X1 = vectorizer.fit_transform(x_train)
X2 = vectorizer.transform(x_test)

In [9]:
X1.toarray()

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

#Model Training

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Initialize and train Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X1, y_train)

# Test the model
predictions = model.predict(X2)
print("Model Accuracy:", accuracy_score(y_test, predictions))

Model Accuracy: 0.8402848423194303


# Simple text to sentiment prediction

In [11]:
a=['i like watermelon, and i am full']
testdata=vectorizer.transform(a)

In [12]:
model.predict(testdata)

array(['positive'], dtype=object)

In [13]:
!pip install SpeechRecognition
!apt-get update
!apt-get install python3-pyaudio portaudio19-dev
!pip install pyaudio

   ---------------------------------------- 0.0/32.9 MB ? eta -:--:--
   ------- -------------------------------- 6.6/32.9 MB 32.0 MB/s eta 0:00:01
   ---------------- ----------------------- 13.6/32.9 MB 32.8 MB/s eta 0:00:01
   ---------------------- ----------------- 18.1/32.9 MB 29.9 MB/s eta 0:00:01
   ---------------------------- ----------- 23.1/32.9 MB 27.2 MB/s eta 0:00:01
   ------------------------------------ --- 29.9/32.9 MB 28.2 MB/s eta 0:00:01
   ---------------------------------------  32.8/32.9 MB 28.4 MB/s eta 0:00:01
   ---------------------------------------- 32.9/32.9 MB 24.5 MB/s eta 0:00:00

   -------------------- ------------------- 2/4 [standard-aifc]
   ------------------------------ --------- 3/4 [SpeechRecognition]
   ------------------------------ --------- 3/4 [SpeechRecognition]
   ------------------------------ --------- 3/4 [SpeechRecognition]
   ------------------------------ --------- 3/4 [SpeechRecognition]
   --------------------------------------

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


In [26]:
import speech_recognition as sr
import threading
import numpy as np

def predict_sentiment(text):
    """Preprocesses input, predicts sentiment, and calculates confidence score."""
    # Preprocess (assumes preprocess_sentence and vectorizer are defined in earlier cells)
    processed_text = preprocess_sentence(text)
    
    # Vectorize
    X_input = vectorizer.transform([processed_text])
    
    # Predict Label
    prediction = model.predict(X_input)[0]
    
    # Calculate Confidence Score (Max probability)
    probabilities = model.predict_proba(X_input)[0]
    confidence = np.max(probabilities) * 100
    
    # Output Display
    print("\n" + "="*40)
    print(f"Recognized Text: '{text}'")
    print(f"Predicted Sentiment: {prediction.upper()}")
    print(f"Confidence Score: {confidence:.2f}%")
    print("="*40 + "\n")

In [27]:
def get_voice_input():
    """Records voice input until user presses ENTER."""
    r = sr.Recognizer()
    spoken_text = []
    stop_recording = False

    def record():
        nonlocal stop_recording
        with sr.Microphone() as source:
            print("\n[Microphone Active] Speak now... press ENTER to stop")
            while not stop_recording:
                try:
                    # Listen for audio chunks
                    audio = r.listen(source, timeout=1, phrase_time_limit=5)
                    text = r.recognize_google(audio)
                    spoken_text.append(text)
                    print("  Heard:", text)
                except:
                    pass # Ignore noise/unrecognized audio

    # Start the recording thread
    t = threading.Thread(target=record)
    t.start()
    
    # Wait for the user to press ENTER in the notebook
    input()   
    
    # Signal the thread to stop and wait for it to finish
    stop_recording = True
    t.join()

    final_text = " ".join(spoken_text)
    return final_text

In [28]:
### --- MAIN INTERFACE LOOP ---
while True:
    print("--- Feedback Sentiment Analyzer ---")
    print("1. Type feedback")
    print("2. Speak feedback (Voice)")
    print("3. Exit")
    
    choice = input("Select an option (1/2/3): ")
    
    if choice == '1':
        user_text = input("\nEnter your feedback: ")
        if user_text.strip():
            predict_sentiment(user_text)
            
    elif choice == '2':
        user_text = get_voice_input()
        if user_text.strip():
            predict_sentiment(user_text)
        else:
            print("\nNo speech detected. Please try again.\n")
            
    elif choice == '3':
        print("\nExiting system...")
        break
    else:
        print("\nInvalid choice. Please enter 1, 2, or 3.\n")

--- Feedback Sentiment Analyzer ---
1. Type feedback
2. Speak feedback (Voice)
3. Exit


Select an option (1/2/3):  2



[Microphone Active] Speak now... press ENTER to stop



No speech detected. Please try again.

--- Feedback Sentiment Analyzer ---
1. Type feedback
2. Speak feedback (Voice)
3. Exit


Select an option (1/2/3):  2


...

Select an option (1/2/3):  2



[Microphone Active] Speak now... press ENTER to stop
  Heard: it's an appreciation
  Heard: for having this kind of know
  Heard: for having this kind of knowledge



Recognized Text: 'it's an appreciation for having this kind of know for having this kind of knowledge'
Predicted Sentiment: POSITIVE
Confidence Score: 80.56%

--- Feedback Sentiment Analyzer ---
1. Type feedback
2. Speak feedback (Voice)
3. Exit


Select an option (1/2/3):  3



Exiting system...


In [30]:
# text -> numeric features
final_text = "Thanks Furqan for this superior knowledge"
X = vectorizer.transform([final_text])

# prediction
prediction = model.predict(X)

print("Prediction:", prediction[0])

Prediction: positive
